# Strategy Tester
Load a fine-tuned model from Hugging Face and benchmark it by sampling trading strategies.

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install git+https://github.com/huggingface/transformers.git@bf3f0ae70d0e902efab4b8517fce88f6697636ce
!pip install --no-deps trl==0.22.2
!pip install backtrader alpaca_trade_api

### Config

In [ ]:
# --- Model ---
BASE_MODEL   = "unsloth/DeepSeek-R1-Distill-Qwen-14B"  # base model the LoRA was trained on
LORA_ADAPTER = None  # HF repo of saved LoRA adapter, e.g. "you/ministral-trading-lora"

# --- Sampling ---
N_SAMPLES      = 20    # strategies to generate
MAX_NEW_TOKENS = 1024
TEMPERATURE    = 1.0

# --- Alpaca & Huggingface ---
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
ALPACA_API_KEY = userdata.get('ALPACA_API_KEY')
ALPACA_SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

# --- Backtest universe (mirrors strategy_generator.ipynb) ---
SYMBOLS = [
    'AAPL', 'AMGN', 'AXP',  'BA',  'CAT',
    'CRM',  'CSCO', 'CVX',  'DIS', 'DOW',
    'GS',   'HD',   'HON',  'IBM', 'JNJ',
    'JPM',  'KO',   'MCD',  'MMM', 'MRK',
    'MSFT', 'NKE',  'NVDA', 'PG',  'TRV',
    'UNH',  'V',    'VZ',   'WBA', 'WMT',
]
START = '2016-01-01'
END   = '2024-12-31'
CASH  = 10_000
TIMEOUT_SECONDS = 10

### Imports

In [ ]:
import json
import os
import re
import random
import shutil
from datetime import datetime

import backtrader as bt
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, display

from unsloth import FastVisionModel, check_python_modules, execute_with_time_limit
from alpaca_trade_api.rest import REST, TimeFrame

### Load Model

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
    fast_inference=False,
)

if LORA_ADAPTER is not None:
    print(f"Loading LoRA adapter: {LORA_ADAPTER}")
    model.load_adapter(LORA_ADAPTER)
    print("Adapter loaded.")
else:
    print("Running base model (no adapter).")

FastVisionModel.for_inference(model)

### Backtrader (singleton + data cache)

In [ ]:
class Backtrader:
    _instance = None
    _data_cache = {}  # shared across all instances, persists for the session

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if hasattr(self, '_initialized'):
            return
        self._initialized = True
        self.rest_api = REST(ALPACA_API_KEY, ALPACA_SECRET_KEY, "https://paper-api.alpaca.markets")
        self.load_bars()

    def _get_bars(self, symbol, timeframe, start, end):
        key = (symbol, str(timeframe), start, end)
        if key not in Backtrader._data_cache:
            Backtrader._data_cache[key] = self.rest_api.get_bars(
                symbol, timeframe, start, end, adjustment='all'
            ).df
        return Backtrader._data_cache[key]

    def load_bars(self, symbols=None, start=None, end=None, timeframe=TimeFrame.Day):
        """Pre-fetch and cache bar data for all symbols before testing."""
        symbols = symbols or SYMBOLS
        start   = start   or START
        end     = end     or END
        print(f"Pre-fetching bar data for {len(symbols)} symbols ({start} → {end})...")
        for i, symbol in enumerate(symbols, 1):
            self._get_bars(symbol, timeframe, start, end)
            print(f"  [{i}/{len(symbols)}] {symbol} ✓")
        print(f"Done. {len(Backtrader._data_cache)} entries cached.")

    def run_backtest(self, strategy, symbols, start, end, timeframe=TimeFrame.Day, cash=10_000):
        cerebro = bt.Cerebro(stdstats=True)
        cerebro.broker.setcash(cash)
        cerebro.addstrategy(strategy)
        cerebro.addanalyzer(bt.analyzers.SharpeRatio,  _name='mysharpe')
        cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='annual_return')
        cerebro.addanalyzer(bt.analyzers.DrawDown,     _name='drawdown')

        if isinstance(symbols, str):
            bars = self._get_bars(symbols, timeframe, start, end)
            cerebro.adddata(bt.feeds.PandasData(dataname=bars, name=symbols))
        else:
            for sym in symbols:
                bars = self._get_bars(sym, timeframe, start, end)
                cerebro.adddata(bt.feeds.PandasData(dataname=bars, name=sym))

        initial = cerebro.broker.getvalue()
        results = cerebro.run()
        final   = cerebro.broker.getvalue()

        strat = results[0]
        return_pct       = (final / initial - 1) * 100
        sharpe_ratio     = strat.analyzers.mysharpe.get_analysis()['sharperatio']
        annual_returns   = strat.analyzers.annual_return.get_analysis()
        avg_annual_return = (sum(annual_returns.values()) / len(annual_returns) * 100) if annual_returns else 0.0
        max_drawdown     = strat.analyzers.drawdown.get_analysis()['max']['drawdown']

        print(f"return={return_pct:.2f}% sharpe={sharpe_ratio} annual={avg_annual_return:.2f}% drawdown={max_drawdown:.2f}%")

        if return_pct > 0:
            cerebro.plot(iplot=False)
            for i, fig_num in enumerate(plt.get_fignums(), start=1):
                plt.figure(fig_num)
                filename = f'backtest_plot_{i}.png'
                plt.savefig(filename, dpi=140, bbox_inches='tight')
                display(Image(filename))
            plt.close('all')

        return return_pct, sharpe_ratio, avg_annual_return, max_drawdown

    @execute_with_time_limit(TIMEOUT_SECONDS)
    def execute_strategy(self, strategy, symbols, start, end, timeframe=TimeFrame.Day, cash=10_000):
        return self.run_backtest(strategy, symbols, start, end, timeframe, cash)


backtrader = Backtrader()

### Helpers

In [ ]:
def make_prompt(symbol: str) -> str:
    return f"""
    Create a trading strategy for {symbol} from {START} to {END} that is fully compatible with the following backtesting setup:

    - Framework: Backtrader
    - Strategy must subclass bt.Strategy
    - The strategy will be passed directly into:
    run_backtest(StrategyClass, symbols, start, end, timeframe, cash)

    STRICT RULES:
    1. Output ONLY a single Python class definition (no explanations, no markdown, no comments outside the class).
    2. The class MUST be named Strategy.
    3. Do NOT include imports (bt is already available).
    4. Do NOT reference external data, files, APIs, or indicators outside Backtrader.
    5. The strategy MUST work for Single-symbol strategies.
    6. All indicators must be created in __init__.
    7. Trading logic must be implemented in next().
    8. Orders must use only:
    - self.buy()
    - self.sell()
    - self.close()
    - self.order_target_percent()
    9. No plotting, printing, logging, or analyzers.
    10. Strategy must be deterministic and backtest-safe (no lookahead bias).

    OUTPUT FORMAT:
    Return ONLY the Python class exactly like this structure:

    class Strategy(bt.Strategy):

        params = dict(
            # parameters here
        )

        def __init__(self):
            # indicator definitions

        def next(self):
            # trading logic

    DO NOT output anything else.
    """.strip()


def generate_strategy_text(symbol: str) -> str:
    prompt = make_prompt(symbol)
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(images=None, text=text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            temperature=TEMPERATURE,
            do_sample=True,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def extract_function(text: str):
    """Extract Strategy class from fenced block or raw text."""
    if text.count("```") >= 2:
        first  = text.find("```") + 3
        second = text.find("```", first)
        fx = text[first:second].strip().removeprefix("python\n")
        fx = fx[fx.find("class Strategy"):]
        if fx.startswith("class Strategy(bt.Strategy):"):
            return fx
    else:
        fx = text.strip()
        fx = fx[fx.find("class Strategy"):]
        if fx.startswith("class Strategy(bt.Strategy):"):
            return fx
    return None


def has_required_functions(text: str) -> bool:
    has_init = bool(re.search(r'def\s+__init__\s*\([^)]*\)\s*:', text or ""))
    has_next = bool(re.search(r'def\s+next\s*\([^)]*\)\s*:', text or ""))
    return has_init and has_next


def function_works(function) -> bool:
    if function is None:
        return False
    ok, info = check_python_modules(function)
    return ok and "error" not in info


def extract_strategy_class(code: str):
    namespace = {'bt': bt}
    exec(code, namespace)
    return namespace['Strategy']


def save_strategy(code, return_pct, sharpe_ratio, avg_annual_return, max_drawdown, symbol):
    """Save a profitable strategy with its stats to successful_strategies/."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    strategy_dir = os.path.join("successful_strategies", f"{timestamp}_{symbol}")
    os.makedirs(strategy_dir, exist_ok=True)

    with open(os.path.join(strategy_dir, "strategy.py"), "w") as f:
        f.write(code)

    stats = {
        "symbol": symbol, "start": START, "end": END,
        "return_pct": round(return_pct, 4),
        "sharpe_ratio": round(sharpe_ratio, 4) if sharpe_ratio is not None else None,
        "avg_annual_return_pct": round(avg_annual_return, 4),
        "max_drawdown_pct": round(max_drawdown, 4),
    }
    with open(os.path.join(strategy_dir, "stats.json"), "w") as f:
        json.dump(stats, f, indent=2)

    for i in range(1, 10):
        src = f"backtest_plot_{i}.png"
        if os.path.exists(src):
            shutil.copy(src, os.path.join(strategy_dir, f"plot_{i}.png"))

    print(f"Saved to {strategy_dir}/")

### Benchmark Loop

In [ ]:
results = []

for i in range(N_SAMPLES):
    symbol = random.choice(SYMBOLS)
    print(f"\n{'='*50}")
    print(f"Sample {i+1}/{N_SAMPLES} | {symbol}  {START} → {END}")

    raw  = generate_strategy_text(symbol)
    code = extract_function(raw)

    record = {
        "sample": i + 1, "symbol": symbol,
        "code": code, "status": None,
        "return_pct": None, "sharpe_ratio": None,
        "avg_annual_return_pct": None, "max_drawdown_pct": None,
    }

    # validate structure first (cheaper), then safety
    if not has_required_functions(code):
        print("FAIL: missing __init__ or next")
        record["status"] = "missing_methods"
        results.append(record)
        continue

    if not function_works(code):
        print("FAIL: invalid/unsafe code")
        record["status"] = "invalid_code"
        results.append(record)
        continue

    try:
        strategy_cls = extract_strategy_class(code)
        return_pct, sharpe_ratio, avg_annual_return, max_drawdown = backtrader.execute_strategy(
            strategy_cls, symbols=symbol, start=START, end=END,
            timeframe=TimeFrame.Day, cash=CASH,
        )

        if return_pct == 0 and sharpe_ratio is None:
            record["status"] = "no_trades"
        else:
            record["status"] = "ok"
            if return_pct > 0:
                save_strategy(code, return_pct, sharpe_ratio, avg_annual_return, max_drawdown, symbol)

        record.update({
            "return_pct": return_pct, "sharpe_ratio": sharpe_ratio,
            "avg_annual_return_pct": avg_annual_return, "max_drawdown_pct": max_drawdown,
        })

    except TimeoutError:
        print("FAIL: timeout")
        record["status"] = "timeout"
    except Exception as e:
        print(f"FAIL: exception — {str(e)[:120]}")
        record["status"] = "exception"

    results.append(record)

print("\nDone.")

### Results Summary

In [ ]:
df = pd.DataFrame(results)

print("=== Status breakdown ===")
print(df["status"].value_counts().to_string())

ok = df[df["status"] == "ok"]
print(f"\nStrategies that ran:    {len(df[df['status'].isin(['ok','no_trades'])])}/{N_SAMPLES}")
print(f"Strategies that traded: {len(ok)}/{N_SAMPLES}")

if len(ok) > 0:
    print("\n=== Stats (strategies that traded) ===")
    metrics = ["return_pct", "avg_annual_return_pct", "sharpe_ratio", "max_drawdown_pct"]
    print(ok[metrics].describe().to_string())
    print(f"\nPositive-return rate:   {(ok['return_pct'] > 0).mean():.1%}")

df.drop(columns=["code"], errors="ignore")

# Save results
model_short = BASE_MODEL.split('/')[-1]
save_dir = f"models/{model_short}"
os.makedirs(save_dir, exist_ok=True)
save_path = f"{save_dir}/Summary.csv"
df.drop(columns=["code"], errors="ignore").to_csv(save_path, index=False)
print(f"\nSaved to {save_path}")

### Plots

In [ ]:
ok = df[df["status"] == "ok"]

if len(ok) == 0:
    print("No successful strategies to plot.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))

    axes[0].hist(ok["return_pct"].dropna(), bins=10, edgecolor="black")
    axes[0].axvline(0, color="red", linestyle="--")
    axes[0].set_title("Total Return (%)")
    axes[0].set_xlabel("Return (%)")

    axes[1].hist(ok["avg_annual_return_pct"].dropna(), bins=10, edgecolor="black", color="seagreen")
    axes[1].axvline(0, color="red", linestyle="--")
    axes[1].set_title("Avg Annual Return (%)")
    axes[1].set_xlabel("Annual Return (%)")

    sharpe_vals = ok["sharpe_ratio"].dropna()
    if len(sharpe_vals) > 0:
        axes[2].hist(sharpe_vals, bins=10, edgecolor="black", color="steelblue")
        axes[2].axvline(0, color="red", linestyle="--")
        axes[2].set_title("Sharpe Ratio")
        axes[2].set_xlabel("Sharpe")
    else:
        axes[2].set_visible(False)

    status_counts = df["status"].value_counts()
    axes[3].pie(status_counts, labels=status_counts.index, autopct="%1.0f%%", startangle=90)
    axes[3].set_title("Outcome Breakdown")

    plt.tight_layout()
    plt.savefig("benchmark_results.png", dpi=140, bbox_inches="tight")
    plt.show()
    print("Saved: benchmark_results.png")

### Top Strategies

In [ ]:
ok_sorted = df[df["status"] == "ok"].sort_values("avg_annual_return_pct", ascending=False)

for _, row in ok_sorted.head(3).iterrows():
    print(f"\n{'='*60}")
    print(f"Sample {row['sample']}  |  {row['symbol']}  {START} → {END}")
    print(f"Return: {row['return_pct']:.2f}%  |  Annual: {row['avg_annual_return_pct']:.2f}%  |  Sharpe: {row['sharpe_ratio']}  |  MaxDD: {row['max_drawdown_pct']:.2f}%")
    print("-" * 60)
    print(row["code"])